# Baby Safety Monitor — Colab 학습

**실행 전 준비사항:**
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. Google Drive에 아래 폴더 구조 업로드:
```
내 드라이브/
└── baby_monitor/
    ├── best.pt          ← ai/runs/baby_monitor_v4/weights/best.pt
    └── dataset_v4/
        ├── data.yaml
        ├── train/
        ├── valid/
        └── test/
```
3. 셀을 순서대로 실행 (런타임 → 모두 실행)

In [ ]:
# 1. 패키지 설치
!pip install ultralytics -q
print('설치 완료')

In [ ]:
# 2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 경로 설정 및 데이터셋 확인
from pathlib import Path
import yaml

DRIVE_BASE = Path('/content/drive/MyDrive/baby_monitor')
DATASET    = DRIVE_BASE / 'dataset_v4'

assert DATASET.exists(), f'dataset_v4 폴더 없음: {DATASET}'

for split in ('train', 'valid', 'test'):
    imgs = list((DATASET / split / 'images').glob('*.*'))
    print(f'  {split}: {len(imgs)}장')

# data.yaml 경로를 Colab 절대경로로 재작성
data_yaml_path = DATASET / 'data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.dump({
        'path' : str(DATASET),
        'train': 'train/images',
        'val'  : 'valid/images',
        'test' : 'test/images',
        'nc'   : 3,
        'names': ['정면', '측면', '후면'],
    }, f, allow_unicode=True, sort_keys=False)
print('data.yaml 업데이트 완료')

In [ ]:
# 4. 학습 실행
from ultralytics import YOLO

existing_model = DRIVE_BASE / 'best.pt'
start_model    = str(existing_model) if existing_model.exists() else 'yolov8n.pt'
print(f'시작 모델: {start_model}')

CONFIG = {
    'data'        : str(data_yaml_path),
    'epochs'      : 200,
    'imgsz'       : 640,
    'batch'       : 16,
    'cache'       : True,
    'patience'    : 20,
    'project'     : str(DRIVE_BASE / 'runs'),
    'name'        : 'baby_monitor_v4',
    'exist_ok'    : True,
    'device'      : 0,
    'workers'     : 2,
    'amp'         : True,
    'cos_lr'      : True,
    'close_mosaic': 10,
    # 증강
    'hsv_h'      : 0.02,
    'hsv_s'      : 0.5,
    'hsv_v'      : 0.5,
    'flipud'     : 0.5,
    'fliplr'     : 0.5,
    'mosaic'     : 1.0,
    'degrees'    : 45.0,
    'translate'  : 0.1,
    'scale'      : 0.5,
    'perspective': 0.0005,
    # 손실 가중치
    'cls': 1.5,
    'box': 7.5,
    'dfl': 1.5,
}

model = YOLO(start_model)
model.train(**CONFIG)
print('\n학습 완료!')

In [ ]:
# 5. 결과 모델을 Drive 루트에 저장
import shutil

best_src = DRIVE_BASE / 'runs' / 'baby_monitor_v4' / 'weights' / 'best.pt'
best_dst = DRIVE_BASE / 'best_new.pt'

if best_src.exists():
    shutil.copy2(best_src, best_dst)
    size_mb = best_src.stat().st_size / 1024 / 1024
    print(f'저장 완료: {best_dst}  ({size_mb:.1f} MB)')
    print('\n다음 단계:')
    print('  Google Drive에서 baby_monitor/best_new.pt 다운로드')
    print('  → 로컬 ai/runs/baby_monitor_v4/weights/best.pt 교체')
else:
    print('[오류] best.pt 없음 — 학습 로그 확인 필요')